# Learning Objectives
---
- To learn about the use of transformers for time series
- To learn about Temporal Fusion Transformers(TFTs)
- To learn about Informers, Autoformers and FEDformers

## Why Transformers for Time Series?
---
In the previous slides, we learned all about the use of RNNs and LSTMs for time series, as well as the limitations of RNN in time series.

That's where transformers come in.

<p align="center">
 <img src="https://i.postimg.cc/pTj28BwN/tf.png" >
 <figcaption align='center'> Fig: Transformer</figcaption>
</p>

As we know, the transformer model is a type of neural network architecture that excels at processing sequential data, most prominently associated with large language models (LLMs). Transformer models have also achieved elite performance in other fields of artificial intelligence (AI), such as computer vision, speech recognition and time series forecasting.

<p align="center">
 <img src="https://i.postimg.cc/bvWnp3x4/transformer.png" >
 <figcaption align='center'> Fig: Transformer Architecture</figcaption>
</p>


## Why Transformers Excel in Time Series tasks:
---
Transformers are better than RNNs in terms of time series analysis in many ways.

- **Parallel Processing**: In transformers, all positions are processed simultaneously, in contrast to RNNs which is a strictly sequential model.

- **Global Attention**: Direct connections between any two time steps

- **Scalability**: RNNs are not able to handle large datasets. Even LSTMs and GRUs, which are modified to handle longer sequences and avoid vanishing gradients, don't perform at the level of the transformer. Transformer has better performance with larger datasets

- **Flexibility**: Transformers can handle multivariate time series naturally, so they would be better than RNN based models to handle time series tasks.

---
## Key Challenges in Time Series Transformers
---
Despite being more well-suited for time-series tasks, you can't just expect a regular transformer to just transition to time series data. Transformers are the best with Natural Language Processing tasks. Unlike NLP however, time series data presents unique challenges:

- **Temporal Ordering and Dependencies**: Unlike text, time series data has patterns that occur over time, making it crucial for forecasting models to understand both short-term and long-term dependencies.

- **Continuous Values**: In time series data we often deal with real-valued sequences, while NLP deals only with discrete tokens.

- **Multiple Horizons**: Many practical applications require forecasting for multiple time steps into the future, not just for the next point.This is known as multi-horizon forecasting.

- **Interpretability**: For sensitive domains like healthcare, finance, and energy, it’s essential to understand how the model arrives at its predictions, enabling transparency and trustworthiness.

- **External Variable Interactions**: Time series data involve dynamic interaction between different variables. Incorporating covariates and features is of huge importance. The importance of each variable may change over time, making variable selection challenging.


:

# Temporal Fusion Transformer (TFT)
---

The **Temporal Fusion Transformer (TFT)** is a Transformer-based model that leverages self-attention to capture complex temporal patterns across multiple time sequences. TFTs are hybrid neural network models combining elements of **LSTM** (Long Short-Term Memory networks), **attention mechanisms**, and a **unique variable selection process**.

A standard transformer assumes its input to be relatively clean and consistent. However, real-world forcasting problems have inputs with **static features**, **known future events**, **observed variables**, all of whom will have different importance at different times.

TFTs use a **gated architecture** to dynamically adjust which features or time steps should influence the forecast, thus improving robustness and interpretability. They were specifically designed for time series forecasting, addressing all those real-world requirements.

It combines the strengths of:
- **LSTMs** → sequential processing & temporal relationships  
- **Attention/Transformers** → long-term dependencies & interpretability  
- **Variable selection networks** → dynamic feature importance  

---





## TFT Architecture

<p align="center">
 <img src="https://i.postimg.cc/SxcTGs1Z/TFTs.jpg" >
 <figcaption align='center'> Fig: TFT Architecture
 <a href="https://arxiv.org/pdf/1912.09363" target="_blank">[Source: TFT paper]</a></figcaption>
</p>

### 1. Inputs

TFT handles three input types:
- **Static covariates** (e.g., product category, location)  
- **Known future inputs** (e.g., holidays, planned prices)  
- **Observed past inputs** (e.g., sales history, weather, demand)


Notation:
- $x^{(s)}$ → static inputs  
- $x^{(k)}_{t}$ → known inputs at time $t$  
- $x^{(o)}_{t}$ → observed inputs at time $t$  

---

### 2. Variable Selection Networks(VSN)

Not all input features are equally useful, so **Variable Selection Networks** act as the first filter. Their purpose is to automatically select relevant features at each time step. At each step, TFT applies **gated residual networks (GRN)** with softmax gating to select relevant variables:
- For static covariates, a static VSN determines which global features matter most.  
- For temporal inputs (past and future), temporal VSNs dynamically reweight features at each time step.

This ensures only useful features influence forecasts and reduces noise from irrelevant features.

---

## 3. Static Covariate Encoders

Static covariate encoders are used to handle time-invariant features (e.g., product category, store location). Static encoders act as **anchors** that guide temporal modeling, ensuring predictions remain consistent with global characteristics. Static features are crucial because they provide **global context**. TFT encodes them into **context vectors** that influence all subsequent computations.  

Static encoders are split into:  
- **Categorical encoders** (embeddings for discrete values like store ID)  
- **Continuous encoders** (transformations for numerical values like capacity)  

Static context vectors are extracted from the encoders. These context vectors are used to:  
- Condition the temporal sequence processing  
- Modulate gating mechanisms  
- Guide the attention module  

This way, static features help the model remain consistent across time.

---

## 4. Local Processing: LSTM Encoder–Decoder

The recurrent backbone of TFT consists of **LSTM encoder-decoder layers**. The **temporal sequence** now flows through an LSTM encoder-decoder:

- **Encoder**: processes observed past observed inputs to capture short-term temporal dependencies
- **Decoder**: processes known future inputs, ensuring the model can project forward with future information  

The hidden states from these LSTMs provide localized temporal representations that are later fused with attention.  

At this point, TFT knows what happened in the past and what is planned for the future, but it still lacks long-term/global understanding.  

---

## 5. Temporal Fusion Decoder (Attention)

To capture long-term dependencies and fuse local with global context, TFT applies a **multi-head self-attention mechanism** to decoder states. It
- Learns which past time steps are most relevant for forecasting  
- Focuses on important future inputs as well  
- Provides interpretability by highlighting key events and lags (via attention weights)

Attention enriches the LSTM backbone, turning local short-term signals into a **globally coherent forecast representation**.  

---

## 6. Gated Residual Networks (GRN)

Throughout TFT, **GRNs** are applied to stabilize and regulate transformations.  
Each GRN:  
- Applies nonlinear transformations  
- Uses gates to allow or block information  
- Normalizes outputs for stable training  

GRNs ensure flexibility without overfitting, while gates act like on/off switches controlling information flow.  

Formula: $$y = LayerNorm(x + Gate(ELU(W_1 x + b_1) W_2 + b_2))$$

---

## 7. Output Layer

Finally, TFT fuses the **attention outputs** with **static context** and passes them through a fully connected layer.  

This produces the multi-horizon forecast: $$\hat{y}_{t+1:t+\tau}$$
where
- $t$ = current time step
- $\tau$ = forecast horizon (number of future steps)





## Advantages of TFT

- **Interpretability**: Unlike many deep learning models, TFT is designed with interpretability in mind. It provides insights at multiple levels:
  - **Variable selection weights**: which features mattered most (static, past, or future)  
  - **Attention weights**: which time steps (historical or future) were most important  
  - **Static context gating**: how static features influenced the forecasts  

- **Flexibility**: TFTs handle mixed multivariate data types (categorical + continuous) as well as missing values

- **Performance**: TFTs produce state-of-the-art results on many benchmarks

- **Practical**: TFTs are designed for real-world decision-making, where trust and explanation are as important as accuracy. They are used when we need uncertainty quantification.



# Informer: Efficient Long Sequence Modeling
---
It's an ordeal for the transformer model to deal with very long time series because they slow down dramatically as the input gets longer. This is due to **quadratic complexity**, which simply means the work the model has to do increases with the square of the input length.

## The Long Sequence Problem
Traditional transformers have $O(L^2)$ complexity for sequence length $L$, making them impractical for very long time series (thousands of time steps). Informer is an efficient long sequence modelimg transformer which deals with this problem.

<p align="center">
 <img src="https://i.postimg.cc/k4P1K2bc/Informer.png" >
 <figcaption align='center'> Fig: Informer Architecture
 <a href="https://www.researchgate.net/figure/Architecture-of-Informer-model_fig4_378008961" target="_blank">[Source: ResearchGate]</a></figcaption>
</p>







## Key Innovation

### 1. **ProbSparse Self-Attention**

Traditional Self-Attention Issues
- Quadratic Complexity: $O(L^2)$ memory and computation
- Redundant Patterns: Many attention patterns are similar
- Information Bottleneck: Not all connections are equally important

ProbSparse Attention Solution
- Core Idea: Not all queries need to attend to all keys, focus on the most informative ones.
- Sparsity Measurement:
$$M(qi, K) = ln(∑e^{(qi·kj/√d)}) - (1/Lk)∑(qi·kj/√d)$$
- Selection Process:
  - Calculate sparsity measure for each query
  - Select top-u queries with highest sparsity

  These queries get full attention, others get reduced attention

Complexity Reduction: $O(L.log(L))$ instead of $O(L^2)$

### 2. **Self-Attention Distillation**

**Problem**: Long sequences create deep networks that are hard to train

**Solution**: Progressive layer reduction

- Stack multiple attention layers
- Apply distilling operation between layers
- Gradually reduce sequence length
- Focuses on dominant attention patterns

## Advantages of Informer

- **Efficiency**: Can handle sequences of length 4000+
- **Memory Optimization**: Significant reduction in memory usage
- **Performance**: Maintains accuracy while being faster
- **Long-term Forecasting**: Excels at long prediction horizons

Owing to the innovations like ProbSparse Self-Attention and Self-Attention Distillation, Informer works well in tasks like predicting electricity usage hours, forecasting traffic trends throughout the day, or modeling financial signals across hundreds of time steps.

It works especially well when there are repeating patterns and large volumes of historical data. However, it may still face challenges when the data is extremely noisy or behaves unpredictably over long periods of the historical data.

# Autoformer: Decomposition-Enhanced Attention
---
While Informer made major strides in handling long time series efficiently, it still treated time as just a sequence of numbers, every time step weighed the same, regardless of whether it was a peak, a seasonal dip, or a long-term trend.

But real-world time series aren't flat, they have structure. For example, sales rise during holidays and electricity demand peaks in the evenings. That's where Autoformer steps in.

## The Decomposition Philosophy
Traditional transformers treat time series as sequences of points. Autoformer recognizes that time series have inherent structure that should be explicitly modeled.

### Time Series Decomposition

Classical Decomposition:
$$X(t) = Trend(t) + Seasonal(t) + Residual(t)$$

Autoformer's Approach:
- Built-in decomposition blocks
- Progressive decomposition through layers
- Separate processing for trend and seasonal components

### Auto-Correlation Mechanism

Problems with Self-Attention for Time Series
- Point-wise Dependencies: Focuses on individual time points
- Ignores Periodicity: Doesn't naturally capture seasonal patterns
- Complex Patterns: Time series have structured dependencies

Auto-Correlation Solution
- Core Idea: Discover period-based dependencies automatically
- Period Detection:
  - Calculate autocorrelation function for the series
  - Identify dominant periods
  - Use these periods to guide attention

Mechanism:

$AutoCorrelation(Q, K, V)$ = Attention based on discovered periods

Benefits:
- Captures seasonal patterns naturally
- Reduces computational complexity
- More interpretable attention patterns

### Progressive Decomposition
**Layer-wise Decomposition**: Each Autoformer layer performs decomposition.
- Trend Extraction: Moving average operation
- Seasonal Processing: Auto-correlation attention on detrended series
- Residual Handling: Remaining components

**Progressive Refinement**:
- Early layers capture coarse patterns
- Deeper layers refine details
- Final prediction combines all components



## Architecture Components

<p align="center">
 <img src="https://i.postimg.cc/y8GwG6Mq/Autoformer.png" >
 <figcaption align='center'> Fig: Autoformer Architecture
 <a href="https://www.researchgate.net/figure/The-architecture-of-the-Autoformer_fig3_391893342" target="_blank">[Source: ResearchGate]</a></figcaption>
</p>

Like other transformers, Autoformer has Encoder-Decoder structure, where encoder process historical data and decoders are used for future prediction generation. There is ross-attention between encoder-decoder.

1. **Auto-Correlation Block**
- Replaces traditional self-attention
- Period-based dependency modeling
- More efficient for time series patterns

2. **Series Decomposition Block**
- Moving average for trend extraction
- Separates trend and seasonal components
- Applied after each attention block

---
## Advantages of Autoformer

- **Interpretability**: Clear trend/seasonal decomposition
- **Efficiency**: Better complexity than standard attention
- **Pattern Recognition**: Natural handling of periodicity
- **Performance**: Strong results on seasonal data

Use Autoformer when there are clear seasonal patterns in data, we need interpretable trend/seasonal decomposition, and the data is periodic business data with medium-length sequences.

# FEDformer: Frequency Enhanced Decomposed Transformer
---
While previous models work in the time domain, FEDformer leverages frequency domain properties of time series.

## Theoretical Foundation
### Fourier Transform for Time Series:
- Time series can be represented in frequency domain
- Different frequencies capture different patterns
- Low frequencies → trends, High frequencies → noise/fine details

### Mixed Attention Philosophy:
- Some patterns better captured in time domain
- Others better captured in frequency domain
- Combine both perspectives for optimal performance

##Frequency Enhanced Block (FEB)
###Fourier Enhanced Attention
Process:
- **Transform to Frequency**: Apply FFT to input sequence $$F(X) = FFT(X)$$
- **Frequency Attention**: Attention in frequency domain $$A_freq = Attention(F(X))$$
- **Inverse Transform**: Convert back to time domain $$X_freq = IFFT(A_{freq})$$
- **Combine**: Merge with time-domain processing $$Output = Combine(X_{time}, X_{freq})$$

### Wavelet Enhanced Attention

Wavelet Transform Benefits:
- Better time-frequency localization than Fourier
- Captures both temporal and spectral features
- Particularly good for non-stationary signals

Process:
- **Wavelet Transform**: Decompose into time-frequency representation
- **Multi-Resolution Attention**: Attention at different scales
- **Inverse Transform**: Reconstruct time series
- **Integration**: Combine with other components


## Architecture Design

<p align="center">
 <img src="https://i.postimg.cc/q75DRmk9/FEDformer-A.png" >
 <figcaption align='center'> Fig: FEDformer Architecture
 <a href="https://www.researchgate.net/figure/The-architecture-of-FEDformer-17_fig1_382587570" target="_blank">[Source: ResearchGate]</a></figcaption>
</p>

1. **Frequency Enhanced Blocks**
- Can use either Fourier or Wavelet transforms
- Separate attention mechanisms for different domains
- Learnable combination weights

2. **Seasonal-Trend Decomposition**
- Similar to Autoformer but enhanced with frequency information
- Frequency domain helps identify seasonal components
- Better trend extraction through low-pass filtering

3. **Multi-Scale Processing**
- Different frequency components processed separately
- Hierarchical attention structure
- Progressive information integration

---
## Mode Selection
### Fourier Mode (FEDformer-f):

- Better for data with clear periodic patterns
- Computationally efficient
- Good for stationary time series

### Wavelet Mode (FEDformer-w):

- Better for non-stationary signals
- Captures local time-frequency features
- More robust to irregular patterns

---
## Advantages of FEDformer

- **Theoretical Soundness**: Solid frequency domain foundation
- **Flexibility**: Choice between Fourier and Wavelet modes
- **Pattern Recognition**: Excellent at capturing periodic patterns
- **Performance**: State-of-the-art results on many benchmarks

Use FEDformer when we are dealing with complex frequency patterns, non-stationary time series, need both time and frequency perspective, or we have domain knowledge about frequency characteristics.

## References

[Temporal Fusion Transformer](https://arxiv.org/pdf/1912.09363)

[Informer](https://arxiv.org/pdf/2012.07436)

[Autoformer](https://arxiv.org/pdf/2106.13008)

[FEDformer](https://arxiv.org/pdf/2201.12740)